In [1]:
import os
import random
import json
import math
import time
from pathlib import Path
from typing import List, Tuple
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, roc_curve, auc
from sklearn.preprocessing import label_binarize
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA

import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import seaborn as sns

print("✓ All libraries imported successfully!")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

✓ All libraries imported successfully!
PyTorch version: 2.8.0+cu126
CUDA available: True


In [2]:
# ========================================
# CONFIGURATION - 4 BAR DATASET
# ========================================

# Dataset configuration
DATASET_NAME = "4bar"
DATA_DIR = r"F:\CP Data\4 bar"
RESULTS_DIR = r"F:\Faisal Work\CP Work\Results"

# Training parameters
SAMPLES_PER_CLASS = 300
IMG_SIZE = 256
BATCH_SIZE = 32
EPOCHS = 40
LEARNING_RATE = 1e-3
SEED = 42

# Augmentation parameters
LABEL_SMOOTHING = 0.05
MIXUP_ALPHA = 0.2
CUTMIX_ALPHA = 0.2
TTA = 8  # Test Time Augmentation

# Other settings
ENSEMBLE_TYPE = "soft"  # "soft" or "meta"
USE_CLASS_WEIGHTS = True
NUM_WORKERS = 0
GRAYSCALE = False
WEAK_AUG = False

print(f"✓ Configuration loaded for dataset: {DATASET_NAME}")
print(f"  Data directory: {DATA_DIR}")
print(f"  Results directory: {RESULTS_DIR}")
print(f"  Samples per class: {SAMPLES_PER_CLASS}")

✓ Configuration loaded for dataset: 4bar
  Data directory: F:\CP Data\4 bar
  Results directory: F:\Faisal Work\CP Work\Results
  Samples per class: 300


In [3]:
def set_seed(seed: int = 42):
    """Set random seeds for reproducibility"""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def device_auto():
    """Automatically select device"""
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")

def count_params(model):
    """Count trainable parameters"""
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def set_plot_style():
    """Set publication-quality plot settings"""
    sns.set_style("white")
    sns.set_context("talk")
    plt.rcParams.update({
        "savefig.dpi": 1000,
        "axes.labelweight": "bold",
        "axes.titleweight": "bold",
        "font.weight": "bold",
        "axes.labelsize": 14,
        "axes.titlesize": 16,
        "xtick.labelsize": 12,
        "ytick.labelsize": 12,
        "legend.fontsize": 12,
        "axes.grid": False
    })

def class_counts_from_indices(dataset, indices):
    """Count samples per class from indices"""
    counts = {}
    for i in indices:
        _, y = dataset[i]
        counts[y] = counts.get(y, 0) + 1
    return counts

def compute_class_weights_from_counts(counts, num_classes):
    """Compute class weights for balanced training"""
    total = sum(counts.get(i, 0) for i in range(num_classes))
    weights = []
    for i in range(num_classes):
        c = counts.get(i, 1)
        w = total / (num_classes * c)
        weights.append(w)
    return torch.tensor(weights, dtype=torch.float32)

def short_labels_from_full(names: List[str]) -> List[str]:
    """Convert full class names to short labels"""
    out = []
    for n in names:
        nl = n.lower()
        if "impeller" in nl:
            out.append("IF")
        elif "hole" in nl:
            out.append("MSH")
        elif "scratch" in nl:
            out.append("MSS")
        elif "normal" in nl:
            out.append("N")
        else:
            out.append(n)
    return out

# Initialize
set_seed(SEED)
set_plot_style()
device = device_auto()

print(f"✓ Utility functions defined")
print(f"  Device: {device}")
print(f"  Seed: {SEED}")

✓ Utility functions defined
  Device: cuda
  Seed: 42


In [4]:
def sample_equal_per_class(dataset, samples_per_class=300, seed=42):
    """
    Sample exactly 'samples_per_class' samples from each class.
    Returns indices of sampled data.
    """
    np.random.seed(seed)
    random.seed(seed)
    
    # Get all indices and their labels
    all_indices = list(range(len(dataset)))
    all_labels = [dataset[i][1] for i in all_indices]
    
    # Group indices by class
    class_indices = {}
    for idx, label in zip(all_indices, all_labels):
        if label not in class_indices:
            class_indices[label] = []
        class_indices[label].append(idx)
    
    num_classes = len(class_indices)
    print(f"\nOriginal class distribution:")
    for cls_id in sorted(class_indices.keys()):
        print(f"  Class {cls_id}: {len(class_indices[cls_id])} samples")
    
    # Sample exactly samples_per_class from each class
    sampled_indices = []
    for cls_id in sorted(class_indices.keys()):
        cls_idx = class_indices[cls_id]
        
        if len(cls_idx) < samples_per_class:
            print(f"\n⚠️  Warning: Class {cls_id} has only {len(cls_idx)} samples")
            print(f"    Using all {len(cls_idx)} samples for this class")
            sampled_indices.extend(cls_idx)
        else:
            selected = np.random.choice(cls_idx, size=samples_per_class, replace=False).tolist()
            sampled_indices.extend(selected)
    
    print(f"\nSampled class distribution:")
    sampled_labels = [dataset[i][1] for i in sampled_indices]
    for cls_id in sorted(class_indices.keys()):
        count = sampled_labels.count(cls_id)
        print(f"  Class {cls_id}: {count} samples")
    
    print(f"\nTotal sampled: {len(sampled_indices)} samples")
    
    return sampled_indices

def stratified_split_from_sampled(dataset, sampled_indices, train_ratio=0.7, val_ratio=0.15, seed=42):
    """
    Create stratified train/val/test split from pre-sampled indices.
    """
    targets = [dataset[i][1] for i in sampled_indices]
    indices_array = np.array(sampled_indices)
    
    # First split: train vs (val + test)
    sss1 = StratifiedShuffleSplit(n_splits=1, test_size=1-train_ratio, random_state=seed)
    train_rel, temp_rel = next(sss1.split(indices_array, targets))
    
    train_idx = indices_array[train_rel]
    temp_idx = indices_array[temp_rel]
    y_temp = np.array(targets)[temp_rel]
    
    # Second split: val vs test
    val_size = int(val_ratio * len(sampled_indices))
    test_size = len(temp_idx) - val_size
    
    sss2 = StratifiedShuffleSplit(n_splits=1, test_size=test_size, random_state=seed)
    val_rel, test_rel = next(sss2.split(temp_idx, y_temp))
    
    val_idx = temp_idx[val_rel]
    test_idx = temp_idx[test_rel]
    
    return train_idx.tolist(), val_idx.tolist(), test_idx.tolist()

print("✓ Data sampling functions defined")

✓ Data sampling functions defined


In [6]:
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, k=3, s=1, p=None, pool=True):
        super().__init__()
        if p is None:
            p = k // 2
        self.conv = nn.Conv2d(in_ch, out_ch, kernel_size=k, stride=s, padding=p, bias=False)
        self.bn = nn.BatchNorm2d(out_ch)
        self.act = nn.ReLU(inplace=True)
        self.pool = nn.MaxPool2d(2) if pool else nn.Identity()
    
    def forward(self, x):
        x = self.act(self.bn(self.conv(x)))
        x = self.pool(x)
        return x

class CNN1_Local(nn.Module):
    """3x3 kernels, 3 conv blocks - Local features"""
    def __init__(self, num_classes=4, in_ch=3, drop=0.2):
        super().__init__()
        self.b1 = ConvBlock(in_ch, 32, k=3)
        self.b2 = ConvBlock(32, 64, k=3)
        self.b3 = ConvBlock(64, 128, k=3)
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Dropout(drop),
            nn.Linear(128, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(drop),
            nn.Linear(128, num_classes)
        )
    
    def forward(self, x):
        return self.head(self.b3(self.b2(self.b1(x))))

class CNN2_Global(nn.Module):
    """5x5 kernels, 4 conv blocks - Global features"""
    def __init__(self, num_classes=4, in_ch=3, drop=0.3):
        super().__init__()
        self.b1 = ConvBlock(in_ch, 32, k=5)
        self.b2 = ConvBlock(32, 64, k=5)
        self.b3 = ConvBlock(64, 128, k=5)
        self.b4 = ConvBlock(128, 256, k=5)
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Dropout(drop),
            nn.Linear(256, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(drop),
            nn.Linear(128, num_classes)
        )
    
    def forward(self, x):
        return self.head(self.b4(self.b3(self.b2(self.b1(x)))))

class DWSeparableConv(nn.Module):
    """Depthwise separable convolution"""
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.dw = nn.Conv2d(in_ch, in_ch, kernel_size=3, stride=stride, padding=1, groups=in_ch, bias=False)
        self.dw_bn = nn.BatchNorm2d(in_ch)
        self.pw = nn.Conv2d(in_ch, out_ch, kernel_size=1, bias=False)
        self.pw_bn = nn.BatchNorm2d(out_ch)
        self.act = nn.ReLU(inplace=True)
    
    def forward(self, x):
        x = self.act(self.dw_bn(self.dw(x)))
        x = self.act(self.pw_bn(self.pw(x)))
        return x

class CNN3_Compact(nn.Module):
    """Depthwise separable (MobileNet-style) - Efficient"""
    def __init__(self, num_classes=4, in_ch=3, drop=0.2):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(in_ch, 32, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True)
        )
        self.blocks = nn.Sequential(
            DWSeparableConv(32, 64, stride=1), nn.MaxPool2d(2),
            DWSeparableConv(64, 128, stride=1), nn.MaxPool2d(2),
            DWSeparableConv(128, 256, stride=1), nn.MaxPool2d(2),
        )
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Dropout(drop),
            nn.Linear(256, num_classes)
        )
    
    def forward(self, x):
        return self.head(self.blocks(self.stem(x)))

class MetaMLP(nn.Module):
    """Meta-learner for ensemble fusion"""
    def __init__(self, num_models=3, num_classes=4, hidden=16, drop=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(num_models * num_classes, hidden),
            nn.ReLU(inplace=True),
            nn.Dropout(drop),
            nn.Linear(hidden, num_classes)
        )
    
    def forward(self, x):
        return self.net(x)

print("✓ Model architectures defined")
print("  - CNN1_Local (3x3 kernels)")
print("  - CNN2_Global (5x5 kernels)")
print("  - CNN3_Compact (Depthwise Separable)")
print("  - MetaMLP (Ensemble fusion)")

✓ Model architectures defined
  - CNN1_Local (3x3 kernels)
  - CNN2_Global (5x5 kernels)
  - CNN3_Compact (Depthwise Separable)
  - MetaMLP (Ensemble fusion)


In [8]:
def build_transforms(img_size=256, grayscale=False, weak_aug=False):
    base = []
    if grayscale:
        base.append(transforms.Grayscale(num_output_channels=1))
    
    train_tf = transforms.Compose(base + [
        transforms.RandomResizedCrop(img_size, scale=(0.8, 1.0)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomVerticalFlip(p=0.1),
        transforms.RandomRotation(10 if not weak_aug else 5),
        transforms.ColorJitter(0.15, 0.15, 0.15, 0.05) if not weak_aug else transforms.ColorJitter(0.05, 0.05, 0.05, 0.02),
        transforms.ToTensor(),
        transforms.Normalize([0.5]*(1 if grayscale else 3), [0.5]*(1 if grayscale else 3)),
    ])
    
    test_tf = transforms.Compose(base + [
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        transforms.Normalize([0.5]*(1 if grayscale else 3), [0.5]*(1 if grayscale else 3)),
    ])
    
    return train_tf, test_tf

def make_loaders(data_dir, img_size=256, batch_size=32, num_workers=0, seed=42, 
                 grayscale=False, weak_aug=False, samples_per_class=300):
    """
    Modified to sample exactly samples_per_class from each class before splitting.
    """
    train_tf, test_tf = build_transforms(img_size, grayscale=grayscale, weak_aug=weak_aug)
    
    # Load full dataset first
    base_full = datasets.ImageFolder(root=data_dir, transform=test_tf)
    
    # Sample equal number from each class
    sampled_indices = sample_equal_per_class(base_full, samples_per_class=samples_per_class, seed=seed)
    
    # Create stratified split from sampled indices
    train_idx, val_idx, test_idx = stratified_split_from_sampled(
        base_full, sampled_indices, train_ratio=0.7, val_ratio=0.15, seed=seed
    )
    
    # Create datasets with appropriate transforms
    base_train = datasets.ImageFolder(root=data_dir, transform=train_tf)
    base_eval = datasets.ImageFolder(root=data_dir, transform=test_tf)
    
    train_ds = Subset(base_train, train_idx)
    val_ds = Subset(base_eval, val_idx)
    test_ds = Subset(base_eval, test_idx)
    
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=num_workers, pin_memory=False)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=False)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=False)
    
    class_to_idx = base_eval.class_to_idx
    idx_to_class = {v: k for k, v in class_to_idx.items()}
    
    return train_loader, val_loader, test_loader, idx_to_class, (train_idx, val_idx, test_idx), base_eval

print("✓ Data loading functions defined")

✓ Data loading functions defined


In [9]:
def rand_bbox(W, H, lam):
    cut_rat = math.sqrt(1. - lam)
    cut_w = int(W * cut_rat)
    cut_h = int(H * cut_rat)
    cx = np.random.randint(W)
    cy = np.random.randint(H)
    x1 = np.clip(cx - cut_w // 2, 0, W)
    y1 = np.clip(cy - cut_h // 2, 0, H)
    x2 = np.clip(cx + cut_w // 2, 0, W)
    y2 = np.clip(cy + cut_h // 2, 0, H)
    return x1, y1, x2, y2

def apply_mixup_cutmix(x, y, mixup_alpha=0.0, cutmix_alpha=0.0):
    if mixup_alpha <= 0 and cutmix_alpha <= 0:
        return x, y, None
    
    lam = 1.0
    if cutmix_alpha > 0 and np.random.rand() < 0.5:
        lam = np.random.beta(cutmix_alpha, cutmix_alpha)
        x2 = x.flip(0)
        W = x.size(3)
        H = x.size(2)
        x1_, y1_, x2_, y2_ = rand_bbox(W, H, lam)
        x[:, :, y1_:y2_, x1_:x2_] = x2[:, :, y1_:y2_, x1_:x2_]
        lam = 1 - ((x2_-x1_) * (y2_-y1_) / (W * H))
        y_a, y_b = y, y.flip(0)
    else:
        lam = np.random.beta(max(1e-8, mixup_alpha), max(1e-8, mixup_alpha))
        x2 = x.flip(0)
        x = lam * x + (1 - lam) * x2
        y_a, y_b = y, y.flip(0)
    
    return x, (y_a, y_b, lam), "mixed"

def loss_with_mixing(criterion, logits, y_or_tuple):
    if isinstance(y_or_tuple, tuple):
        y_a, y_b, lam = y_or_tuple
        return lam * criterion(logits, y_a) + (1 - lam) * criterion(logits, y_b)
    return criterion(logits, y_or_tuple)

print("✓ Mixup/CutMix functions defined")

✓ Mixup/CutMix functions defined


In [11]:
def train_one_model(model, train_loader, val_loader, epochs, lr, device, ckpt_path,
                    class_weights=None, label_smoothing=0.0, mixup_alpha=0.0, cutmix_alpha=0.0):
    model.to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(1, epochs))
    criterion = nn.CrossEntropyLoss(
        weight=class_weights.to(device) if class_weights is not None else None,
        label_smoothing=label_smoothing
    )
    
    best_val = float('inf')
    best_state = None
    patience = 10
    no_imp = 0
    
    for ep in range(1, epochs + 1):
        model.train()
        tr_loss = 0
        tr_correct = 0
        n = 0
        
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            x, y_mix, mixed_flag = apply_mixup_cutmix(x, y, mixup_alpha=mixup_alpha, cutmix_alpha=cutmix_alpha)
            
            opt.zero_grad()
            logits = model(x)
            loss = loss_with_mixing(criterion, logits, y_mix if mixed_flag else y)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            opt.step()
            
            tr_loss += loss.item() * x.size(0)
            tr_correct += (logits.argmax(1) == y).sum().item()
            n += x.size(0)
        
        sched.step()
        
        # Validation
        model.eval()
        vl_loss = 0
        vl_correct = 0
        m = 0
        
        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(device), y.to(device)
                logits = model(x)
                loss = criterion(logits, y)
                vl_loss += loss.item() * x.size(0)
                vl_correct += (logits.argmax(1) == y).sum().item()
                m += x.size(0)
        
        tr_loss /= max(1, n)
        tr_acc = tr_correct / max(1, n)
        vl_loss /= max(1, m)
        vl_acc = vl_correct / max(1, m)
        
        print(f"Epoch {ep:03d} | train loss {tr_loss:.4f} acc {tr_acc:.3f} | val loss {vl_loss:.4f} acc {vl_acc:.3f}")
        
        if vl_loss < best_val:
            best_val = vl_loss
            best_state = {k: v.detach().cpu() for k, v in model.state_dict().items()}
            torch.save(best_state, ckpt_path)
            no_imp = 0
        else:
            no_imp += 1
            if no_imp >= patience:
                print("Early stopping.")
                break
    
    if best_state is not None:
        model.load_state_dict(best_state)
    
    return model

print("✓ Training function defined")

✓ Training function defined


In [12]:
@torch.no_grad()
def predict_probs(model, loader, device, tta=1) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    model.eval()
    all_logits = []
    all_probs = []
    all_labels = []
    
    for x, y in loader:
        x = x.to(device)
        
        if tta <= 1:
            logits = model(x)
            probs = F.softmax(logits, dim=1).cpu().numpy()
        else:
            probs_accum = 0
            for i in range(tta):
                x_aug = x
                if i % 2 == 1:
                    x_aug = torch.flip(x_aug, dims=[3])  # horizontal flip
                if i % 4 == 2:
                    x_aug = torch.flip(x_aug, dims=[2])  # vertical flip
                logits = model(x_aug)
                probs_accum += F.softmax(logits, dim=1)
            probs = (probs_accum / tta).cpu().numpy()
            logits = model(x).cpu().numpy()
        
        all_logits.append(logits if tta <= 1 else model(x).cpu().numpy())
        all_probs.append(probs)
        all_labels.append(y.numpy())
    
    return np.vstack(all_logits), np.vstack(all_probs), np.concatenate(all_labels)

def soft_vote(probs_list: List[np.ndarray]) -> np.ndarray:
    return np.mean(np.stack(probs_list, axis=0), axis=0)

def train_meta_mlp(val_probs_list, val_labels, num_classes=4, epochs=100, lr=1e-3, hidden=16, device=None):
    X = np.concatenate(val_probs_list, axis=1)
    y = val_labels
    X_t = torch.tensor(X, dtype=torch.float32, device=device)
    y_t = torch.tensor(y, dtype=torch.long, device=device)
    
    model = MetaMLP(num_models=len(val_probs_list), num_classes=num_classes, hidden=hidden, drop=0.1).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    criterion = nn.CrossEntropyLoss()
    
    best_loss = float('inf')
    best_state = None
    patience = 15
    no_imp = 0
    
    for ep in range(1, epochs + 1):
        model.train()
        opt.zero_grad()
        logits = model(X_t)
        loss = criterion(logits, y_t)
        loss.backward()
        opt.step()
        
        with torch.no_grad():
            vl_loss = loss.item()
        
        if vl_loss < best_loss:
            best_loss = vl_loss
            best_state = {k: v.detach().cpu() for k, v in model.state_dict().items()}
            no_imp = 0
        else:
            no_imp += 1
            if no_imp >= patience:
                break
    
    if best_state is not None:
        model.load_state_dict(best_state)
    
    model.eval()
    return model

print("✓ Inference and evaluation functions defined")

✓ Inference and evaluation functions defined


In [13]:
def save_confusion_matrix(figpath, cm, labels):
    set_plot_style()
    plt.figure(figsize=(8, 6))
    ax = sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                     xticklabels=labels, yticklabels=labels, cbar=False,
                     annot_kws={"size": 20, "fontweight": "bold"})
    ax.set_xlabel('Predicted Label', fontsize=16, fontweight='bold')
    ax.set_ylabel('True Label', fontsize=16, fontweight='bold')
    plt.setp(ax.get_xticklabels(), fontweight='bold', fontsize=14, rotation=45, ha='right')
    plt.setp(ax.get_yticklabels(), fontweight='bold', fontsize=14)
    plt.tight_layout()
    plt.savefig(figpath, dpi=1000, bbox_inches='tight')
    plt.close()
    print(f"  ✓ Saved: {figpath}")

def save_roc_curve(figpath, y_true, probs, labels):
    set_plot_style()
    K = len(labels)
    y_bin = label_binarize(y_true, classes=list(range(K)))
    
    plt.figure(figsize=(10, 8))
    
    # Plot ROC curve for each class
    for i in range(K):
        fpr, tpr, _ = roc_curve(y_bin[:, i], probs[:, i])
        roc_auc = auc(fpr, tpr)
        plt.plot(fpr, tpr, lw=3, label=f'{labels[i]} (AUC = {roc_auc:.2f})')
    
    # Diagonal reference line (random classifier)
    plt.plot([0, 1], [0, 1], color='navy', lw=3, linestyle='--', label='Random Classifier')
    
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate', fontsize=16, fontweight='bold')
    plt.ylabel('True Positive Rate', fontsize=16, fontweight='bold')
    plt.legend(loc='lower right', fontsize=12, frameon=True, shadow=True)
    plt.grid(alpha=0.3, linestyle='--', linewidth=0.5)
    plt.tight_layout()
    plt.savefig(figpath, dpi=1000, bbox_inches='tight')
    plt.close()
    print(f"  ✓ Saved: {figpath}")

print("✓ Visualization functions (Part 1) defined")

✓ Visualization functions (Part 1) defined


In [14]:
def save_tsne(figpath, features, y_true, labels):
    """2D t-SNE visualization"""
    set_plot_style()
    N = features.shape[0]
    perplexity = max(5, min(30, (N - 1) // 3))
    
    tsne = TSNE(n_components=2, init="pca", random_state=42,
                perplexity=perplexity, learning_rate="auto")
    xy = tsne.fit_transform(features)
    
    markers = ['o', 's', '^', 'v']
    colors = ['blue', 'red', 'green', 'purple']
    
    plt.figure(figsize=(12, 10))
    for i, cname in enumerate(labels):
        sel = (y_true == i)
        plt.scatter(xy[sel, 0], xy[sel, 1],
                    marker=markers[i % len(markers)],
                    color=colors[i % len(colors)],
                    label=cname, alpha=0.7, s=30)
    
    plt.legend(title="Classes", loc='upper right',
               prop={'weight': 'bold', 'size': 12}, title_fontsize=13)
    plt.xlabel('t-SNE Component 1', fontsize=14, fontweight='bold')
    plt.ylabel('t-SNE Component 2', fontsize=14, fontweight='bold')
    plt.xticks(fontsize=12, fontweight='bold')
    plt.yticks(fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig(figpath, dpi=1000, bbox_inches='tight')
    plt.close()
    print(f"  ✓ Saved: {figpath}")

def save_3d_scatter(figpath, features, y_true, labels):
    """3D scatter plot using PCA for dimensionality reduction"""
    set_plot_style()
    
    # Use PCA for 3D reduction
    pca = PCA(n_components=3, random_state=42)
    coords_3d = pca.fit_transform(features)
    
    # Color scheme
    colors = ['green', 'orange', 'blue', 'red']  # IF, MSH, MSS, Normal
    markers = ['s', 's', 's', 's']  # All squares
    
    fig = plt.figure(figsize=(12, 10))
    ax = fig.add_subplot(111, projection='3d')
    
    # Plot each class
    for i, cname in enumerate(labels):
        sel = (y_true == i)
        ax.scatter(coords_3d[sel, 0], 
                  coords_3d[sel, 1], 
                  coords_3d[sel, 2],
                  c=colors[i % len(colors)],
                  marker=markers[i % len(markers)],
                  label=cname,
                  alpha=0.7,
                  s=50,
                  edgecolors='black',
                  linewidth=0.5)
    
    # Set labels
    ax.set_xlabel('UMAP1', fontsize=14, fontweight='bold', labelpad=10)
    ax.set_ylabel('UMAP2', fontsize=14, fontweight='bold', labelpad=10)
    ax.set_zlabel('UMAP3', fontsize=14, fontweight='bold', labelpad=10)
    
    # Legend
    ax.legend(loc='upper right', fontsize=12, frameon=True, 
              prop={'weight': 'bold'})
    
    # Grid
    ax.grid(True, alpha=0.3, linestyle='--', linewidth=0.5)
    ax.xaxis.pane.fill = False
    ax.yaxis.pane.fill = False
    ax.zaxis.pane.fill = False
    
    # Set viewing angle
    ax.view_init(elev=20, azim=45)
    
    plt.tight_layout()
    plt.savefig(figpath, dpi=1000, bbox_inches='tight')
    plt.close()
    print(f"  ✓ Saved: {figpath}")

def save_classification_report(filepath, y_true, y_pred, labels):
    report = classification_report(y_true, y_pred, target_names=labels, digits=4, zero_division=0)
    acc = accuracy_score(y_true, y_pred)
    
    with open(filepath, 'w', encoding='utf-8') as f:
        f.write(f"Overall Accuracy: {acc:.4f}\n\n")
        f.write("="*70 + "\n")

In [15]:
# Create output directories
dataset_result_dir = Path(RESULTS_DIR) / DATASET_NAME
checkpoints_dir = dataset_result_dir / "checkpoints"
figures_dir = dataset_result_dir / "figures"
reports_dir = dataset_result_dir / "reports"

for d in [checkpoints_dir, figures_dir, reports_dir]:
    d.mkdir(parents=True, exist_ok=True)

print("✓ Output directories created:")
print(f"  - {checkpoints_dir}")
print(f"  - {figures_dir}")
print(f"  - {reports_dir}")

✓ Output directories created:
  - F:\Faisal Work\CP Work\Results\4bar\checkpoints
  - F:\Faisal Work\CP Work\Results\4bar\figures
  - F:\Faisal Work\CP Work\Results\4bar\reports


In [16]:
print("="*80)
print(f"CP FAULT DIAGNOSIS - {DATASET_NAME.upper()}")
print(f"EQUAL SAMPLING: {SAMPLES_PER_CLASS} samples per class")
print("="*80)

# Load data with equal sampling
print("\nLoading data with equal sampling per class...")
train_loader, val_loader, test_loader, idx_to_class, (train_idx, val_idx, test_idx), base_eval = make_loaders(
    DATA_DIR, img_size=IMG_SIZE, batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS, seed=SEED,
    grayscale=GRAYSCALE, weak_aug=WEAK_AUG,
    samples_per_class=SAMPLES_PER_CLASS
)

num_classes = len(idx_to_class)
in_ch = 1 if GRAYSCALE else 3
class_names_full = [idx_to_class[i] for i in range(num_classes)]
class_names_short = short_labels_from_full(class_names_full)

print(f"\nClasses: {class_names_short}")
print(f"Train: {len(train_idx)} | Val: {len(val_idx)} | Test: {len(test_idx)}")

# Verify class distribution in splits
print("\nClass distribution in splits:")
for split_name, split_idx in [("Train", train_idx), ("Val", val_idx), ("Test", test_idx)]:
    counts = class_counts_from_indices(base_eval, split_idx)
    print(f"  {split_name}: {counts}")

# Class weights
class_w = None
if USE_CLASS_WEIGHTS:
    counts = class_counts_from_indices(base_eval, train_idx)
    class_w = compute_class_weights_from_counts(counts, num_classes)
    print(f"\nClass weights: {[f'{w:.3f}' for w in class_w.tolist()]}")

CP FAULT DIAGNOSIS - 4BAR
EQUAL SAMPLING: 300 samples per class

Loading data with equal sampling per class...

Original class distribution:
  Class 0: 376 samples
  Class 1: 376 samples
  Class 2: 351 samples
  Class 3: 478 samples
  Class 4: 351 samples
  Class 5: 478 samples
  Class 6: 325 samples
  Class 7: 325 samples

Sampled class distribution:
  Class 0: 300 samples
  Class 1: 300 samples
  Class 2: 300 samples
  Class 3: 300 samples
  Class 4: 300 samples
  Class 5: 300 samples
  Class 6: 300 samples
  Class 7: 300 samples

Total sampled: 2400 samples

Classes: ['IF', 'IF', 'MSH', 'MSS', 'MSH', 'MSS', 'N', 'N']
Train: 1679 | Val: 360 | Test: 361

Class distribution in splits:
  Train: {0: 210, 3: 210, 2: 210, 7: 210, 4: 210, 1: 210, 6: 209, 5: 210}
  Val: {2: 45, 3: 45, 1: 45, 6: 45, 0: 45, 4: 45, 5: 45, 7: 45}
  Test: {7: 45, 3: 45, 4: 45, 0: 45, 5: 45, 1: 45, 6: 46, 2: 45}

Class weights: ['0.999', '0.999', '0.999', '0.999', '0.999', '0.999', '1.004', '0.999']


In [17]:
print("\nBuilding ensemble models...")
m1 = CNN1_Local(num_classes=num_classes, in_ch=in_ch)
m2 = CNN2_Global(num_classes=num_classes, in_ch=in_ch)
m3 = CNN3_Compact(num_classes=num_classes, in_ch=in_ch)

print(f"CNN1_Local params: {count_params(m1):,}")
print(f"CNN2_Global params: {count_params(m2):,}")
print(f"CNN3_Compact params: {count_params(m3):,}")


Building ensemble models...
CNN1_Local params: 111,016
CNN2_Global params: 1,112,488
CNN3_Compact params: 49,352


In [18]:
print("\n" + "="*80)
print("TRAINING CNN1_LOCAL")
print("="*80)

m1 = train_one_model(m1, train_loader, val_loader, EPOCHS, LEARNING_RATE, device,
                     checkpoints_dir / "cnn1_local.pt",
                     class_weights=class_w, label_smoothing=LABEL_SMOOTHING,
                     mixup_alpha=MIXUP_ALPHA, cutmix_alpha=CUTMIX_ALPHA)

print("✓ CNN1_Local training completed!")


TRAINING CNN1_LOCAL
Epoch 001 | train loss 1.7414 acc 0.223 | val loss 1.5633 acc 0.250
Epoch 002 | train loss 1.6157 acc 0.270 | val loss 1.4325 acc 0.311
Epoch 003 | train loss 1.6069 acc 0.239 | val loss 1.3573 acc 0.411
Epoch 004 | train loss 1.5683 acc 0.244 | val loss 1.3468 acc 0.375
Epoch 005 | train loss 1.6082 acc 0.267 | val loss 1.3642 acc 0.414
Epoch 006 | train loss 1.5771 acc 0.294 | val loss 1.2704 acc 0.400
Epoch 007 | train loss 1.5306 acc 0.310 | val loss 1.3115 acc 0.364
Epoch 008 | train loss 1.5514 acc 0.301 | val loss 1.2484 acc 0.475
Epoch 009 | train loss 1.5020 acc 0.370 | val loss 1.4042 acc 0.353
Epoch 010 | train loss 1.4763 acc 0.306 | val loss 1.2331 acc 0.472
Epoch 011 | train loss 1.5241 acc 0.291 | val loss 1.2012 acc 0.503
Epoch 012 | train loss 1.4853 acc 0.347 | val loss 1.0858 acc 0.594
Epoch 013 | train loss 1.4316 acc 0.361 | val loss 1.3385 acc 0.331
Epoch 014 | train loss 1.3726 acc 0.394 | val loss 1.0399 acc 0.594
Epoch 015 | train loss 1.38

In [19]:
print("\n" + "="*80)
print("TRAINING CNN2_GLOBAL")
print("="*80)

m2 = train_one_model(m2, train_loader, val_loader, EPOCHS, LEARNING_RATE, device,
                     checkpoints_dir / "cnn2_global.pt",
                     class_weights=class_w, label_smoothing=LABEL_SMOOTHING,
                     mixup_alpha=MIXUP_ALPHA, cutmix_alpha=CUTMIX_ALPHA)

print("✓ CNN2_Global training completed!")


TRAINING CNN2_GLOBAL
Epoch 001 | train loss 1.7427 acc 0.219 | val loss 1.3892 acc 0.411
Epoch 002 | train loss 1.6049 acc 0.291 | val loss 1.3724 acc 0.386
Epoch 003 | train loss 1.5158 acc 0.301 | val loss 1.5617 acc 0.311
Epoch 004 | train loss 1.5116 acc 0.326 | val loss 1.2155 acc 0.467
Epoch 005 | train loss 1.4141 acc 0.373 | val loss 1.1883 acc 0.481
Epoch 006 | train loss 1.3491 acc 0.396 | val loss 0.9734 acc 0.614
Epoch 007 | train loss 1.2828 acc 0.350 | val loss 1.8034 acc 0.306
Epoch 008 | train loss 1.3044 acc 0.356 | val loss 1.1658 acc 0.503
Epoch 009 | train loss 1.3163 acc 0.387 | val loss 0.9727 acc 0.614
Epoch 010 | train loss 1.3558 acc 0.367 | val loss 1.3880 acc 0.394
Epoch 011 | train loss 1.3043 acc 0.378 | val loss 1.6483 acc 0.297
Epoch 012 | train loss 1.2558 acc 0.426 | val loss 0.9417 acc 0.633
Epoch 013 | train loss 1.2920 acc 0.443 | val loss 1.2191 acc 0.472
Epoch 014 | train loss 1.2262 acc 0.401 | val loss 0.9696 acc 0.656
Epoch 015 | train loss 1.2

In [20]:
print("\n" + "="*80)
print("TRAINING CNN3_COMPACT")
print("="*80)

m3 = train_one_model(m3, train_loader, val_loader, EPOCHS, LEARNING_RATE, device,
                     checkpoints_dir / "cnn3_compact.pt",
                     class_weights=class_w, label_smoothing=LABEL_SMOOTHING,
                     mixup_alpha=MIXUP_ALPHA, cutmix_alpha=CUTMIX_ALPHA)

print("✓ CNN3_Compact training completed!")


TRAINING CNN3_COMPACT
Epoch 001 | train loss 1.7285 acc 0.197 | val loss 1.4503 acc 0.325
Epoch 002 | train loss 1.5655 acc 0.314 | val loss 1.2413 acc 0.536
Epoch 003 | train loss 1.4826 acc 0.373 | val loss 1.1350 acc 0.622
Epoch 004 | train loss 1.3990 acc 0.426 | val loss 1.0325 acc 0.639
Epoch 005 | train loss 1.3798 acc 0.346 | val loss 1.1188 acc 0.531
Epoch 006 | train loss 1.3712 acc 0.366 | val loss 1.1490 acc 0.542
Epoch 007 | train loss 1.3180 acc 0.385 | val loss 0.9306 acc 0.658
Epoch 008 | train loss 1.2413 acc 0.454 | val loss 0.9282 acc 0.639
Epoch 009 | train loss 1.2727 acc 0.411 | val loss 0.9121 acc 0.653
Epoch 010 | train loss 1.2505 acc 0.341 | val loss 0.9524 acc 0.628
Epoch 011 | train loss 1.2540 acc 0.404 | val loss 0.9364 acc 0.647
Epoch 012 | train loss 1.2325 acc 0.471 | val loss 0.8971 acc 0.664
Epoch 013 | train loss 1.2209 acc 0.426 | val loss 0.9193 acc 0.647
Epoch 014 | train loss 1.1804 acc 0.473 | val loss 0.9290 acc 0.644
Epoch 015 | train loss 1.

In [21]:
print("\n" + "="*80)
print("GENERATING PREDICTIONS WITH TTA")
print("="*80)

print(f"\nRunning inference with TTA={TTA}...")

# Validation predictions
v1_logits, v1, yv = predict_probs(m1.to(device), val_loader, device, tta=TTA)
v2_logits, v2, _ = predict_probs(m2.to(device), val_loader, device, tta=TTA)
v3_logits, v3, _ = predict_probs(m3.to(device), val_loader, device, tta=TTA)

# Test predictions
t1_logits, t1, yt = predict_probs(m1.to(device), test_loader, device, tta=TTA)
t2_logits, t2, _ = predict_probs(m2.to(device), test_loader, device, tta=TTA)
t3_logits, t3, _ = predict_probs(m3.to(device), test_loader, device, tta=TTA)

print("✓ Predictions generated successfully!")
print(f"  Validation samples: {len(yv)}")
print(f"  Test samples: {len(yt)}")


GENERATING PREDICTIONS WITH TTA

Running inference with TTA=8...
✓ Predictions generated successfully!
  Validation samples: 360
  Test samples: 361


In [22]:
print("\n" + "="*80)
print("ENSEMBLE FUSION")
print("="*80)

if ENSEMBLE_TYPE == "soft":
    print("\nUsing soft voting ensemble...")
    val_fused = soft_vote([v1, v2, v3])
    test_fused = soft_vote([t1, t2, t3])
    test_logits_fused = soft_vote([t1_logits, t2_logits, t3_logits])
else:
    print("\nTraining meta-learner...")
    meta = train_meta_mlp([v1, v2, v3], yv, num_classes=num_classes, device=device)
    
    X_val = np.concatenate([v1, v2, v3], axis=1)
    with torch.no_grad():
        val_logits = meta(torch.tensor(X_val, dtype=torch.float32, device=device)).cpu().numpy()
    val_fused = F.softmax(torch.tensor(val_logits), dim=1).numpy()
    
    X_test = np.concatenate([t1, t2, t3], axis=1)
    with torch.no_grad():
        test_logits = meta(torch.tensor(X_test, dtype=torch.float32, device=device)).cpu().numpy()
    test_fused = F.softmax(torch.tensor(test_logits), dim=1).numpy()
    test_logits_fused = test_logits
    
    torch.save(meta.state_dict(), checkpoints_dir / "meta_mlp.pt")

print("✓ Ensemble fusion completed!")


ENSEMBLE FUSION

Using soft voting ensemble...
✓ Ensemble fusion completed!


In [23]:
print("\n" + "="*80)
print("EVALUATION & VISUALIZATION")
print("="*80)

preds_ensemble = test_fused.argmax(1)
cm_ensemble = confusion_matrix(yt, preds_ensemble)

print("\nGenerating visualizations...")

# Confusion Matrix
save_confusion_matrix(
    figures_dir / f"{DATASET_NAME}_confusion_matrix.png",
    cm_ensemble,
    class_names_short
)

# ROC Curve
save_roc_curve(
    figures_dir / f"{DATASET_NAME}_roc_curve.png",
    yt,
    test_fused,
    class_names_short
)

# 2D t-SNE
save_tsne(
    figures_dir / f"{DATASET_NAME}_tsne_2d.png",
    test_logits_fused,
    yt,
    class_names_short
)

# 3D Scatter Plot
save_3d_scatter(
    figures_dir / f"{DATASET_NAME}_3d_scatter.png",
    test_logits_fused,
    yt,
    class_names_short
)

# Classification Report
acc = save_classification_report(
    reports_dir / f"{DATASET_NAME}_classification_report.txt",
    yt,
    preds_ensemble,
    class_names_short
)

print("\n✓ All visualizations generated!")


EVALUATION & VISUALIZATION

Generating visualizations...
  ✓ Saved: F:\Faisal Work\CP Work\Results\4bar\figures\4bar_confusion_matrix.png
  ✓ Saved: F:\Faisal Work\CP Work\Results\4bar\figures\4bar_roc_curve.png
  ✓ Saved: F:\Faisal Work\CP Work\Results\4bar\figures\4bar_tsne_2d.png
  ✓ Saved: F:\Faisal Work\CP Work\Results\4bar\figures\4bar_3d_scatter.png

✓ All visualizations generated!


In [24]:
# Save metadata
metadata = {
    "dataset_name": DATASET_NAME,
    "data_dir": DATA_DIR,
    "samples_per_class": SAMPLES_PER_CLASS,
    "num_classes": num_classes,
    "class_names": class_names_short,
    "train_samples": len(train_idx),
    "val_samples": len(val_idx),
    "test_samples": len(test_idx),
    "img_size": IMG_SIZE,
    "epochs": EPOCHS,
    "ensemble_type": ENSEMBLE_TYPE,
    "tta": TTA,
    "test_accuracy": float(acc),
    "seed": SEED
}

with open(reports_dir / f"{DATASET_NAME}_metadata.json", 'w') as f:
    json.dump(metadata, f, indent=2)

# Save split indices
with open(checkpoints_dir / "split_indices.json", 'w') as f:
    json.dump({"train": train_idx, "val": val_idx, "test": test_idx}, f, indent=2)

with open(checkpoints_dir / "idx_to_class.json", 'w', encoding='utf-8') as f:
    json.dump(idx_to_class, f, indent=2, ensure_ascii=False)

print("✓ Metadata and indices saved!")

TypeError: float() argument must be a string or a real number, not 'NoneType'

In [25]:
print("\n" + "="*80)
print("COMPLETED SUCCESSFULLY!")
print("="*80)
print(f"\n✓ Test Accuracy: {acc:.4f}")
print(f"✓ Samples per class: {SAMPLES_PER_CLASS}")
print(f"✓ Results saved to: {dataset_result_dir}")
print(f"\n  📁 Checkpoints: {checkpoints_dir}")
print(f"  📊 Figures: {figures_dir}")
print(f"  📄 Reports: {reports_dir}")
print("\n" + "="*80)

# Display results summary
print("\n📈 RESULTS SUMMARY:")
print(f"  Dataset: {DATASET_NAME}")
print(f"  Classes: {', '.join(class_names_short)}")
print(f"  Test Accuracy: {acc:.4f} ({acc*100:.2f}%)")
print(f"  Training Epochs: {EPOCHS}")
print(f"  TTA Augmentations: {TTA}")
print(f"  Ensemble Type: {ENSEMBLE_TYPE}")


COMPLETED SUCCESSFULLY!


TypeError: unsupported format string passed to NoneType.__format__